In [ ]:
import torch
import torch.nn as nn
import sys
import os
import time
import threading
import shutil

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
# Define SimAM module
class SimAM(nn.Module):
    """SimAM: Parameter-Free Attention (ICML 2021)
    Zero learnable parameters = 100% pretrained weights preserved.
    """
    def __init__(self, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.act = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = w * h - 1
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)) + 0.5
        return x * self.act(y)

In [ ]:
# Register SimAM with ultralytics
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as modules

for cls in [SimAM]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s):
            setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls

torch.serialization.add_safe_globals([SimAM])
print("SimAM registered!")

In [ ]:
# UPDATE THIS PATH to your local dataset location
DATASET = "path/to/your/helmet-detection-dataset/archive7_extracted"

# Create data.yaml
content = f"""train: {DATASET}/images/train
val: {DATASET}/images/val
test: {DATASET}/images/test
nc: 2
names: ['helmet', 'non-helmet']
"""
with open("data.yaml", "w") as f:
    f.write(content)

# SimAM YAML architecture (3 placements: after SPPF, P4, P5)
yaml_text = """nc: 2
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 1, SimAM, []]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, SimAM, []]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [-1, 1, SimAM, []]
  - [[16, 20, 24], 1, Detect, [nc]]
"""
with open("simam.yaml", "w") as f:
    f.write(yaml_text)

print("Both YAML files created!")

In [ ]:
from ultralytics import YOLO

# ========================
# PATHS (centralized - modified for local)
# ========================
RUN_NAME = "simam_3layer"
WORK_DIR = f"runs/{RUN_NAME}"
OUTPUT_DIR = f"output/{RUN_NAME}"

# ========================
# BACKUP THREAD (SMART) - Optional for local training
# ========================
# def backup_loop():
#     weights_src = os.path.join(WORK_DIR, "weights")
#     weights_dst = os.path.join(OUTPUT_DIR, "weights")
#     while True:
#         if os.path.exists(weights_src):
#             try:
#                 shutil.copytree(weights_src, weights_dst, dirs_exist_ok=True)
#                 print("✅ Weights backup updated")
#             except Exception as e:
#                 print("Backup error:", e)
#         time.sleep(300)  # every 5 min
# threading.Thread(target=backup_loop, daemon=True).start()

# ========================
# MODEL
# ========================
model = YOLO("simam.yaml")
model.load("yolov8n.pt")

# ========================
# TRAINING (UNCHANGED, added device=0)
# ========================
model.train(
    data="data.yaml",
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer="SGD",
    lr0=0.01,
    cos_lr=True,
    amp=True,
    close_mosaic=10,
    patience=50,
    workers=2,
    save=True,
    save_period=10,
    project="runs",
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    device=0  # RTX 4060
)

print("TRAINING DONE!")

# ========================
# FINAL SAVE (FULL RUN)
# ========================
if os.path.exists(WORK_DIR):
    shutil.copytree(WORK_DIR, OUTPUT_DIR, dirs_exist_ok=True)
    print(f"✅ Full run saved to {OUTPUT_DIR} (weights + graphs + curves + logs)")
else:
    print("❌ Run directory not found")